In [25]:
from dotenv import load_dotenv
load_dotenv()


True

In [34]:
def llm(prompt):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=150
    )
    response = response.choices[0].message.content
    return response.output_text

In [35]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [36]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [37]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [30]:
a1=len(documents)
print(a1, "documents read from the repository.")

72 documents read from the repository.


In [31]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

In [38]:
q2="how does the agentic loop keep calling the model until it stops?"
results = index.search(q2, num_results=1)
a2=results[0]['filename']
print("the filename of the first result is ",a2)

the filename of the first result is  01-agentic-rag/lessons/14-agentic-loop.md


In [33]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py

--2026-06-22 21:53:06--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.111.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 

200 OK
Length: 2134 (2.1K) [text/plain]
Saving to: ‘rag_helper.py.1’

rag_helper.py.1     100%[===================>]   2.08K  --.-KB/s    in 0s      

2026-06-22 21:53:06 (23.8 MB/s) - ‘rag_helper.py.1’ saved [2134/2134]



In [42]:
import os
from openai import OpenAI
from dotenv import load_dotenv

# Load .env
load_dotenv()

# Initialize OpenRouter
client = OpenAI(
    base_url=os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1"),
    api_key=os.getenv("OPENROUTER_API_KEY"),
)
model = os.getenv("OPENROUTER_MODEL", "openai/gpt-4o-mini")

class RAGBase:
    """Adapted RAGBase for document schema with OpenRouter support."""
    
    def __init__(self, index, client, model, max_results=5):
        self.index = index
        self.client = client
        self.model = model
        self.max_results = max_results
    
    def search(self, query):
        """Search using content field."""
        return self.index.search(query, num_results=self.max_results)
    
    def build_context(self, results):
        """Build context using filename/content."""
        context = ""
        for result in results:
            context += f"File: {result.get('filename', 'Unknown')}\n"
            context += f"Content: {result.get('content', '')}\n\n"
        return context
    
    def rag(self, query):
        """RAG that returns (answer, input_tokens)."""
        # Search and build context
        results = self.search(query)
        context = self.build_context(results)
        
        # Create prompt
        prompt = f"""Context from course lessons:

{context}

Question: {query}

Answer based on the context:"""
        
        # Get response from OpenRouter
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", "content": "You are a helpful course assistant."},
                {"role": "user", "content": prompt}
            ],
            extra_headers={
                "HTTP-Referer": "http://localhost",
                "X-Title": "LLM Zoomcamp Homework",
            }
        )
        
        return response.choices[0].message.content, response.usage.prompt_tokens

# Usage
rag = RAGBase(index, client, model)
answer, input_tokens = rag.rag("How does the agentic loop keep calling the model until it stops?")
print(f"Input tokens: {input_tokens}")

Input tokens: 7104


In [43]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [44]:
a4=len(chunks)
print(a4, "chunks")

295 chunks


In [45]:
import os
from openai import OpenAI
from dotenv import load_dotenv
import minsearch
from gitsource import chunk_documents

# Load .env
load_dotenv()

# Initialize OpenRouter
client = OpenAI(
    base_url=os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1"),
    api_key=os.getenv("OPENROUTER_API_KEY"),
)
model = os.getenv("OPENROUTER_MODEL", "openai/gpt-4o-mini")

# Your RAGBase class (same as Q3)
class RAGBase:
    """Adapted RAGBase for document schema with OpenRouter support."""
    
    def __init__(self, index, client, model, max_results=5):
        self.index = index
        self.client = client
        self.model = model
        self.max_results = max_results
    
    def search(self, query):
        """Search using content field."""
        return self.index.search(query, num_results=self.max_results)
    
    def build_context(self, results):
        """Build context using filename/content."""
        context = ""
        for result in results:
            context += f"File: {result.get('filename', 'Unknown')}\n"
            context += f"Content: {result.get('content', '')}\n\n"
        return context
    
    def rag(self, query):
        """RAG that returns (answer, input_tokens)."""
        results = self.search(query)
        context = self.build_context(results)
        
        prompt = f"""Context from course lessons:

{context}

Question: {query}

Answer based on the context:"""
        
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", "content": "You are a helpful course assistant."},
                {"role": "user", "content": prompt}
            ],
            extra_headers={
                "HTTP-Referer": "http://localhost",
                "X-Title": "LLM Zoomcamp Homework",
            }
        )
        
        return response.choices[0].message.content, response.usage.prompt_tokens

# ============================================================================
# Q5: RAG with Chunked Documents
# ============================================================================

print("="*60)
print("Q5: RAG with Chunked Documents")
print("="*60)

# You already have these from Q4:
# chunks = chunk_documents(documents, size=2000, step=1000)
# a4 = len(chunks)

# Index the chunks
print("Indexing chunks...")
chunk_index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
chunk_index.fit(chunks)

# Initialize RAG with chunk index
rag_chunked = RAGBase(chunk_index, client, model, max_results=5)

# Run the same query as Q3
query = "How does the agentic loop keep calling the model until it stops?"
print(f"\nQuery: {query}")
print("-"*60)

answer_chunked, input_tokens_chunked = rag_chunked.rag(query)

print(f"Answer: {answer_chunked}")
print(f"\nInput tokens with chunking: {input_tokens_chunked}")

# ============================================================================
# Compare with Q3 results
# ============================================================================

# You should have saved these from Q3
# Let's assume you saved them or re-run Q3
print("\n" + "="*60)
print("Token Comparison")
print("="*60)

# Re-run Q3 to get fresh token count for comparison
print("Re-running Q3 to get token count for comparison...")
rag_full = RAGBase(index, client, model, max_results=5)  # index from Q2 (full documents)
answer_full, input_tokens_full = rag_full.rag(query)

print(f"Full documents (Q3): {input_tokens_full} input tokens")
print(f"Chunked documents (Q5): {input_tokens_chunked} input tokens")

# Calculate reduction ratio
reduction_ratio = input_tokens_full / input_tokens_chunked

print(f"\nReduction ratio: {reduction_ratio:.1f}x fewer")

# Determine the answer option
if reduction_ratio > 20:
    reduction_desc = "30× fewer"
elif reduction_ratio > 6:
    reduction_desc = "10× fewer"
elif reduction_ratio > 2:
    reduction_desc = "3× fewer"
else:
    reduction_desc = "about the same"

print(f"\nQ5 Answer: {reduction_desc}")
print(f"  (Full: {input_tokens_full} tokens → Chunked: {input_tokens_chunked} tokens)")

Q5: RAG with Chunked Documents
Indexing chunks...

Query: How does the agentic loop keep calling the model until it stops?
------------------------------------------------------------
Answer: The agentic loop continuously calls the model until it receives a response without any function calls. This is achieved through a `while True` loop that iterates as long as the model indicates it still requires additional actions.

Here's how it works step by step:

1. **Initialization**: The loop starts with an iteration counter (`it`) and a flag (`has_function_calls`) that tracks whether the model requests any tool (function) calls.

2. **Model Call**: Within each iteration, the model is called with the message history (which includes instructions and user questions) using the `openai_client.responses.create` method.

3. **Handling the Response**: The response from the model is evaluated:
   - If the response contains items of type `function_call`, the agent executes the specified function (`mak

In [48]:
import os
from openai import OpenAI
from dotenv import load_dotenv
import minsearch
from gitsource import chunk_documents
import json
from typing import List, Dict, Any

# Load .env
load_dotenv()

# Initialize OpenRouter
client = OpenAI(
    base_url=os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1"),
    api_key=os.getenv("OPENROUTER_API_KEY"),
)
model = os.getenv("OPENROUTER_MODEL", "openai/gpt-4o-mini")

# ============================================================================
# Q6: Custom Agent Implementation (No ToyAIKit)
# ============================================================================

print("="*60)
print("Q6: Agentic RAG with Custom Agent")
print("="*60)

# Use the chunk_index from Q5
# chunk_index should already be defined and fitted with chunks

def search_function(query: str) -> List[Dict[str, Any]]:
    """
    Search the course lessons for information about the query.
    
    Args:
        query: The search query string
        
    Returns:
        List of relevant chunks with content and source information
    """
    results = chunk_index.search(query, num_results=5)
    return [
        {
            "content": r.get("content", "")[:500],  # Truncate for readability
            "filename": r.get("filename", "Unknown"),
            "score": r.get("score", 0.0)
        }
        for r in results
    ]

class SimpleAgent:
    """
    Simple agent that can use tools in a loop.
    """
    
    def __init__(self, client, model, instructions, tools, max_iterations=10):
        self.client = client
        self.model = model
        self.instructions = instructions
        self.tools = tools
        self.max_iterations = max_iterations
        self.tool_calls = []
    
    def run(self, query):
        messages = [
            {"role": "system", "content": self.instructions},
            {"role": "user", "content": query}
        ]
        
        iteration = 0
        tool_results = []
        
        while iteration < self.max_iterations:
            iteration += 1
            
            # Get response from LLM
            response = self.client.chat.completions.create(
                model=self.model,
                messages=messages,
                temperature=0.0
            )
            
            response_text = response.choices[0].message.content
            
            # Check if the agent wants to use a tool
            # Look for patterns like "search: query" or "[SEARCH] query"
            import re
            
            # Search for tool calls in the response
            search_patterns = [
                r'search(?:\s*):\s*["\']?(.*?)["\']?(?:\n|$)',
                r'\[SEARCH\](.*?)(?:\n|$)',
                r'using search:\s*(.*?)(?:\n|$)',
                r'search for\s*["\']?(.*?)["\']?(?:\n|$)',
            ]
            
            found_tool = False
            
            for pattern in search_patterns:
                matches = re.findall(pattern, response_text, re.IGNORECASE)
                if matches:
                    for match in matches:
                        query_text = match.strip()
                        if query_text and len(query_text) > 3:  # Avoid empty or too short queries
                            print(f"  Agent searching: {query_text}")
                            self.tool_calls.append(query_text)
                            
                            # Execute the search
                            results = search_function(query_text)
                            tool_results.append({
                                "query": query_text,
                                "results": results
                            })
                            
                            # Add tool result to messages
                            messages.append({
                                "role": "assistant",
                                "content": response_text
                            })
                            messages.append({
                                "role": "user",
                                "content": f"Search results for '{query_text}':\n{json.dumps(results, indent=2)}"
                            })
                            
                            found_tool = True
                            break
                if found_tool:
                    break
            
            # If no tool was called, the agent is done
            if not found_tool:
                return response_text
        
        # If we hit max iterations, return the last response
        return response_text

# Track search calls
search_calls = []

def tracked_search(query: str) -> List[Dict[str, Any]]:
    """Wrapper to track search calls."""
    search_calls.append(query)
    return search_function(query)

# Create the agent
print("\nCreating custom agent with search tool...")

agent = SimpleAgent(
    client=client,
    model=model,
    instructions="""You're a course teaching assistant for the LLM Zoomcamp.
Answer the student's question using the search tool.
Make multiple searches with different keywords before answering.
Be thorough and provide comprehensive answers based on the course material.

When you want to search, respond with: "search: your query here"
You can make multiple searches. Do this until you have enough information to answer.""",
    tools=[],  # We're using a simpler approach
    max_iterations=10
)

# Run the agentic query
query_q6 = "How does the agentic loop work, and how is it different from plain RAG?"
print(f"\nQuery: {query_q6}")
print("-"*60)

result = agent.run(query_q6)

print(f"\nAgent Answer: {result}")
print("\n" + "="*60)
print("Search Statistics")
print("="*60)

# Count search calls
search_count = len(search_calls)

print(f"Number of search calls: {search_count}")
print(f"Search queries used:")
for i, q in enumerate(search_calls, 1):
    print(f"  {i}. {q}")

# Determine the answer option
if search_count >= 20:
    count_desc = "20"
elif search_count >= 10:
    count_desc = "10"
elif search_count >= 4:
    count_desc = "4"
else:
    count_desc = "0"

print(f"\nQ6 Answer: {count_desc} search calls")

Q6: Agentic RAG with Custom Agent

Creating custom agent with search tool...

Query: How does the agentic loop work, and how is it different from plain RAG?
------------------------------------------------------------
  Agent searching: agentic loop in LLMs"

Agent Answer: Based on the search results, here’s a more detailed explanation of the **agentic loop** and how it differs from **RAG (Retrieval-Augmented Generation)**:

### Agentic Loop

The **agentic loop** is a framework used in AI agents, particularly in the context of language models (LLMs). It involves a continuous cycle where the model generates outputs, evaluates them, and potentially makes further tool calls based on those outputs. This process is often implemented using a `while` loop that keeps running until a satisfactory final answer is produced, meaning the model has no further tool calls to make.

1. **Structure**: The agentic loop typically consists of:
   - Generating a response or output.
   - Evaluating that outp